# 01 — Bronze Layer Ingestion

**Medallion Tier 1.** Land the raw CSVs as **Parquet** under `Files/Bronze/`, preserving the original
structure (no business transformation). Professional touches:
- **Explicit string schema** (no inference) so columns are never silently coerced/dropped.
- **Audit columns**: `_ingested_at`, `_source_file`, `_batch_id` for lineage.
- **Partitioned by `_load_date`** for efficient incremental landing.
- **Run logged** to `audit_pipeline_run_log`; failures raise and are recorded.

> The brief requires a **Data Factory pipeline** for the Copy step — see
> `pipeline/bronze_copy_pipeline.json` and `pipeline/master_orchestration_pipeline.json`.
> This notebook is the code-equivalent and what the master pipeline invokes for repeatability.

In [ ]:
# --- Parameters (overridable by the orchestration pipeline) ---
batch_id = ""        # empty -> generated below
reprocess = True      # True = overwrite this batch's Bronze output

In [ ]:
%run 00_common_utils

In [ ]:
batch_id = batch_id or new_batch_id()
log(f"Bronze batch_id = {batch_id}")

def land_to_bronze(entity: str):
    src = CONFIG["sources"][entity]
    run = start_run("bronze", entity, batch_id)
    try:
        reader = spark.read.option("inferSchema", "false")
        for k, v in src["csv_options"].items():
            reader = reader.option(k, v)
        df = reader.csv(src["raw_file"])  # all columns land as string
        df = (df
            .withColumn("_ingested_at", F.current_timestamp())
            .withColumn("_source_file", F.lit(src["raw_file"]))
            .withColumn("_batch_id", F.lit(batch_id))
            .withColumn("_load_date", F.current_date()))
        n = df.count()
        mode = "overwrite" if reprocess else "append"
        (df.write.mode(mode).partitionBy("_load_date").parquet(src["bronze_path"]))
        end_run(run, "SUCCESS", rows_read=n, rows_written=n)
        log(f"bronze/{entity}: {n} rows -> {src['bronze_path']}")
    except Exception as e:
        end_run(run, "FAILED", message=str(e))
        raise

for entity in CONFIG["sources"]:
    land_to_bronze(entity)

In [ ]:
# Validation — structure preserved, audit columns present
for entity, src in CONFIG["sources"].items():
    df = spark.read.parquet(src["bronze_path"])
    print(f"\n=== Bronze/{entity}: {df.count()} rows, {len(df.columns)} cols ===")
    df.printSchema()
    df.show(3, truncate=70)

In [ ]:
import notebookutils
notebookutils.notebook.exit(batch_id)  # hand the batch_id to the next stage